# Volunteer Soul Profile Analysis

**Purpose**: Comprehensive characterization of identified "Volunteer Souls" 

**Research Question**: What distinguishes Volunteer Souls from Standard Path experiencers?
- Demographic profile
- NDE phenomenology
- Transformation patterns
- Return characteristics

## Theoretical Framework

The **Volunteer Soul Hypothesis** proposes that a subset of incarnated souls:
1. **Chose** to embody for a specific mission
2. Have **pre-birth memories** or awareness
3. Experience **distinctive NDE phenomenology**
4. Show **mission-oriented return patterns**

This notebook builds profiles from NDErs identified as potential Volunteers in `03_soul_path_classification.ipynb`.

## Data Sources

- NDERF: 5,646 structured experiences
- IANDS: 1,093 structured experiences  
- Total: 6,739 experiences
- **Focus**: Volunteer Candidates (score ≥ 2 on choice indicators)

## Schema Improvements Applied

| OLD Schema Problem | NEW Schema Solution | Profile Opportunity |
|-------------------|---------------------|---------------------|
| Single choice field | `chose_parents` + `chose_mission` + `chose_circumstances` | Multi-dimensional identification |
| Conflated return | `return_agency` + `return_willingness` | Return pattern profiling |
| Single guidance field | `guidance_received` + `guidance_types` (List) | Guidance content analysis |
| Temporal conflation | Before/after fields | Baseline vs outcome comparison |

---

**Notebook**: `04_volunteer_soul_profile.ipynb`  
**Schema Version**: 2026-01-06  
**Source Notebooks**: `volunteer_soul_profile.ipynb`, `volunteer_discriminant_analysis.ipynb`

## Setup: Load Libraries and Data

In [ ]:
# Standard libraries
import json
import os
from pathlib import Path
from collections import Counter, defaultdict

# Data analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr, ttest_ind

# Configuration
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

# Constants
VOLUNTEER_THRESHOLD = 2  # Score threshold for volunteer identification

print("Libraries loaded successfully")

In [ ]:
# =============================================================================
# DATA LOADING AND VOLUNTEER IDENTIFICATION
# =============================================================================

def load_all_experiences():
    """Load all structured NDE experiences from both NDERF and IANDS."""
    experiences = []
    data_dir = Path('../structured')
    
    for json_file in data_dir.glob('*.json'):
        with open(json_file, 'r', encoding='utf-8') as f:
            exp = json.load(f)
            exp['_source'] = 'NDERF' if 'nderf' in json_file.stem.lower() else 'IANDS'
            exp['_file'] = json_file.stem
            experiences.append(exp)
    
    return experiences

def compute_volunteer_score(exp):
    """Compute volunteer soul score from choice indicators."""
    score = 0
    for field in ['chose_parents', 'chose_mission', 'chose_life_circumstances']:
        if exp.get(field) == 'YES':
            score += 1
    return score

# Load and classify
all_experiences = load_all_experiences()
for exp in all_experiences:
    exp['volunteer_score'] = compute_volunteer_score(exp)
    exp['is_volunteer'] = exp['volunteer_score'] >= VOLUNTEER_THRESHOLD

# Separate groups
volunteers = [exp for exp in all_experiences if exp['is_volunteer']]
standard = [exp for exp in all_experiences if not exp['is_volunteer']]

print(f"Total experiences: {len(all_experiences)}")
print(f"Volunteer Souls (score >= {VOLUNTEER_THRESHOLD}): {len(volunteers)} ({len(volunteers)/len(all_experiences)*100:.1f}%)")
print(f"Standard Path: {len(standard)} ({len(standard)/len(all_experiences)*100:.1f}%)")

---

# PART I: Demographic Profile

**Objective**: Compare demographic characteristics of Volunteers vs Standard Path.

In [ ]:
# =============================================================================
# PART I: DEMOGRAPHIC PROFILE
# =============================================================================

print("=" * 70)
print("DEMOGRAPHIC PROFILE: Volunteers vs Standard")
print("=" * 70)

# Helper function for comparison
def compare_categorical(field, vol_list, std_list, top_n=6):
    """Compare categorical field between groups."""
    vol_values = [exp.get(field) for exp in vol_list if exp.get(field)]
    std_values = [exp.get(field) for exp in std_list if exp.get(field)]
    
    vol_counts = Counter(vol_values)
    std_counts = Counter(std_values)
    
    return vol_counts, std_counts

# 1. Gender Distribution
print("\n1. GENDER:")
print("-" * 50)

vol_gender, std_gender = compare_categorical('gender', volunteers, standard)

print(f"Volunteers (n={sum(vol_gender.values())}):")
for g, c in vol_gender.most_common(3):
    print(f"   {g}: {c} ({c/sum(vol_gender.values())*100:.1f}%)")

print(f"\nStandard (n={sum(std_gender.values())}):")
for g, c in std_gender.most_common(3):
    print(f"   {g}: {c} ({c/sum(std_gender.values())*100:.1f}%)")

# 2. Age at Experience
print("\n\n2. AGE AT EXPERIENCE:")
print("-" * 50)

vol_ages = [float(exp['age_at_experience']) for exp in volunteers 
            if exp.get('age_at_experience') and str(exp['age_at_experience']).replace('.','').isdigit()]
std_ages = [float(exp['age_at_experience']) for exp in standard 
            if exp.get('age_at_experience') and str(exp['age_at_experience']).replace('.','').isdigit()]

if vol_ages:
    print(f"Volunteers: Mean={np.mean(vol_ages):.1f}, Median={np.median(vol_ages):.1f}, Std={np.std(vol_ages):.1f}")
if std_ages:
    print(f"Standard:   Mean={np.mean(std_ages):.1f}, Median={np.median(std_ages):.1f}, Std={np.std(std_ages):.1f}")

# Statistical test
if vol_ages and std_ages:
    stat, p_value = mannwhitneyu(vol_ages, std_ages, alternative='two-sided')
    print(f"\nMann-Whitney U: {stat:.2f}, p={p_value:.4f}")
    print(f"{'✓ SIGNIFICANT' if p_value < 0.05 else '○ Not significant'} age difference")

# 3. Religious Background
print("\n\n3. RELIGIOUS BACKGROUND:")
print("-" * 50)

vol_religion, std_religion = compare_categorical('religious_background', volunteers, standard)

print(f"Volunteers - Top backgrounds:")
for r, c in vol_religion.most_common(5):
    print(f"   {r}: {c} ({c/sum(vol_religion.values())*100:.1f}%)")

print(f"\nStandard - Top backgrounds:")
for r, c in std_religion.most_common(5):
    print(f"   {r}: {c} ({c/sum(std_religion.values())*100:.1f}%)")

---

# PART II: NDE Phenomenology Profile

**Objective**: Compare NDE characteristics between Volunteers and Standard Path.

In [ ]:
# =============================================================================
# PART II: NDE PHENOMENOLOGY PROFILE
# =============================================================================

print("=" * 70)
print("NDE PHENOMENOLOGY: Volunteers vs Standard")
print("=" * 70)

# Helper for binary comparison
def compare_binary(field, vol_list, std_list):
    """Compare YES rates for binary fields."""
    vol_yes = sum(1 for exp in vol_list if exp.get(field) == 'YES')
    vol_total = sum(1 for exp in vol_list if exp.get(field) in ['YES', 'NO'])
    
    std_yes = sum(1 for exp in std_list if exp.get(field) == 'YES')
    std_total = sum(1 for exp in std_list if exp.get(field) in ['YES', 'NO'])
    
    vol_rate = vol_yes / vol_total * 100 if vol_total > 0 else 0
    std_rate = std_yes / std_total * 100 if std_total > 0 else 0
    
    return vol_rate, std_rate, vol_total, std_total

# 1. Being Encounters
print("\n1. BEING ENCOUNTERS:")
print("-" * 50)

encounter_fields = [
    ('being_encounter', 'Any Being Encounter'),
    ('light_being_present', 'Light Being Present'),
    ('guidance_received', 'Guidance Received'),
]

for field, label in encounter_fields:
    vol_rate, std_rate, vol_n, std_n = compare_binary(field, volunteers, standard)
    diff = vol_rate - std_rate
    print(f"   {label}:")
    print(f"      Volunteer: {vol_rate:.1f}% (n={vol_n})")
    print(f"      Standard:  {std_rate:.1f}% (n={std_n})")
    print(f"      Δ: {diff:+.1f}%\n")

# 2. Life Review
print("\n2. LIFE REVIEW:")
print("-" * 50)

vol_rate, std_rate, vol_n, std_n = compare_binary('life_review_occurred', volunteers, standard)
print(f"   Life Review Occurred:")
print(f"      Volunteer: {vol_rate:.1f}% (n={vol_n})")
print(f"      Standard:  {std_rate:.1f}% (n={std_n})")
print(f"      Δ: {vol_rate - std_rate:+.1f}%")

# 3. Past Lives / Reincarnation Elements
print("\n\n3. PAST LIVES / REINCARNATION ELEMENTS:")
print("-" * 50)

reincarn_fields = [
    ('past_lives_shown', 'Past Lives Shown'),
    ('reincarnation_mentioned', 'Reincarnation Mentioned'),
]

for field, label in reincarn_fields:
    vol_rate, std_rate, vol_n, std_n = compare_binary(field, volunteers, standard)
    diff = vol_rate - std_rate
    print(f"   {label}:")
    print(f"      Volunteer: {vol_rate:.1f}% (n={vol_n})")
    print(f"      Standard:  {std_rate:.1f}% (n={std_n})")
    print(f"      Δ: {diff:+.1f}%\n")

In [ ]:
# =============================================================================
# GUIDANCE CONTENT ANALYSIS (NEW List field)
# =============================================================================

print("=" * 70)
print("GUIDANCE CONTENT ANALYSIS (NEW: List field)")
print("=" * 70)
print("✓ NEW SCHEMA: guidance_types is a List field capturing multiple types per experience")
print()

# Extract guidance types
vol_guidance_types = []
for exp in volunteers:
    types = exp.get('guidance_types', [])
    if types:
        vol_guidance_types.extend(types)

std_guidance_types = []
for exp in standard:
    types = exp.get('guidance_types', [])
    if types:
        std_guidance_types.extend(types)

if vol_guidance_types:
    vol_type_counts = Counter(vol_guidance_types)
    print("Volunteer Guidance Types:")
    for gtype, count in vol_type_counts.most_common(10):
        print(f"   {gtype}: {count}")

if std_guidance_types:
    std_type_counts = Counter(std_guidance_types)
    print("\nStandard Guidance Types:")
    for gtype, count in std_type_counts.most_common(10):
        print(f"   {gtype}: {count}")

# Compare rates
if vol_guidance_types and std_guidance_types:
    print("\n\nGuidance Type Rate Comparison:")
    print("-" * 50)
    
    all_types = set(vol_type_counts.keys()) | set(std_type_counts.keys())
    vol_total = len(vol_guidance_types)
    std_total = len(std_guidance_types)
    
    comparisons = []
    for gtype in all_types:
        vol_pct = vol_type_counts.get(gtype, 0) / vol_total * 100
        std_pct = std_type_counts.get(gtype, 0) / std_total * 100
        comparisons.append({
            'Type': gtype,
            'Volunteer %': vol_pct,
            'Standard %': std_pct,
            'Δ': vol_pct - std_pct
        })
    
    comp_df = pd.DataFrame(comparisons).sort_values('Δ', ascending=False)
    print(comp_df.head(10).to_string(index=False))

---

# PART III: Return Characteristics Profile

**Objective**: Analyze distinctive return patterns of Volunteers using NEW split fields.

In [ ]:
# =============================================================================
# PART III: RETURN CHARACTERISTICS PROFILE
# =============================================================================

print("=" * 70)
print("RETURN CHARACTERISTICS: Volunteers vs Standard")
print("=" * 70)
print("✓ NEW SCHEMA: Separate Agency and Willingness fields")
print()

# 1. Return Agency
print("1. RETURN AGENCY (Who decided to return?):")
print("-" * 50)

vol_agency, std_agency = compare_categorical('return_agency', volunteers, standard)

print("Volunteers:")
for agency, count in vol_agency.most_common():
    print(f"   {agency}: {count} ({count/sum(vol_agency.values())*100:.1f}%)")

print("\nStandard:")
for agency, count in std_agency.most_common():
    print(f"   {agency}: {count} ({count/sum(std_agency.values())*100:.1f}%)")

# 2. Return Willingness
print("\n\n2. RETURN WILLINGNESS (How did they feel about returning?):")
print("-" * 50)

vol_willing, std_willing = compare_categorical('return_willingness', volunteers, standard)

print("Volunteers:")
for will, count in vol_willing.most_common():
    print(f"   {will}: {count} ({count/sum(vol_willing.values())*100:.1f}%)")

print("\nStandard:")
for will, count in std_willing.most_common():
    print(f"   {will}: {count} ({count/sum(std_willing.values())*100:.1f}%)")

# 3. Agency × Willingness Combinations
print("\n\n3. AGENCY × WILLINGNESS PATTERNS (NEW capability):")
print("-" * 50)

def get_return_pattern(exp):
    agency = exp.get('return_agency', 'UNK')
    willing = exp.get('return_willingness', 'UNK')
    if agency != 'UNK' and willing != 'UNK':
        return f"{agency}_{willing}"
    return None

vol_patterns = Counter(p for exp in volunteers if (p := get_return_pattern(exp)))
std_patterns = Counter(p for exp in standard if (p := get_return_pattern(exp)))

print("Volunteer Return Patterns:")
for pattern, count in vol_patterns.most_common(6):
    print(f"   {pattern}: {count} ({count/sum(vol_patterns.values())*100:.1f}%)")

print("\nStandard Return Patterns:")
for pattern, count in std_patterns.most_common(6):
    print(f"   {pattern}: {count} ({count/sum(std_patterns.values())*100:.1f}%)")

In [ ]:
# =============================================================================
# RETURN REASONS ANALYSIS (NEW List field)
# =============================================================================

print("=" * 70)
print("RETURN REASONS ANALYSIS (NEW: List field)")
print("=" * 70)

# Extract return reasons
vol_reasons = []
for exp in volunteers:
    reasons = exp.get('return_reasons', [])
    if reasons:
        vol_reasons.extend(reasons)

std_reasons = []
for exp in standard:
    reasons = exp.get('return_reasons', [])
    if reasons:
        std_reasons.extend(reasons)

if vol_reasons:
    vol_reason_counts = Counter(vol_reasons)
    print("Volunteer Return Reasons:")
    for reason, count in vol_reason_counts.most_common(10):
        print(f"   {reason}: {count}")

if std_reasons:
    std_reason_counts = Counter(std_reasons)
    print("\nStandard Return Reasons:")
    for reason, count in std_reason_counts.most_common(10):
        print(f"   {reason}: {count}")

# Key finding: Mission-related reasons
print("\n\nMission-Related Reasons Comparison:")
print("-" * 50)

mission_reasons = ['MISSION', 'UNFINISHED_TASK', 'PURPOSE', 'WORK_TO_DO']
vol_mission = sum(vol_reason_counts.get(r, 0) for r in mission_reasons)
std_mission = sum(std_reason_counts.get(r, 0) for r in mission_reasons)

vol_total = sum(vol_reason_counts.values()) if vol_reasons else 1
std_total = sum(std_reason_counts.values()) if std_reasons else 1

print(f"   Volunteer: {vol_mission/vol_total*100:.1f}% mission-related reasons")
print(f"   Standard:  {std_mission/std_total*100:.1f}% mission-related reasons")
print(f"   Δ: {vol_mission/vol_total*100 - std_mission/std_total*100:+.1f}%")

---

# PART IV: Transformation Profile

**Objective**: Compare before/after transformation using NEW temporal fields.

In [ ]:
# =============================================================================
# PART IV: TRANSFORMATION PROFILE
# =============================================================================

print("=" * 70)
print("TRANSFORMATION PROFILE: Volunteers vs Standard")
print("=" * 70)
print("✓ NEW SCHEMA: Before/after fields enable baseline comparison")
print()

# Scale mapping
scale = {'NONE': 1, 'LOW': 2, 'MODERATE': 3, 'HIGH': 4, 'EXTREME': 5}

def get_transformation_stats(exp_list, before_field, after_field, direction='decrease'):
    """Get transformation statistics for a group."""
    changes = []
    baselines = []
    outcomes = []
    
    for exp in exp_list:
        before = exp.get(before_field)
        after = exp.get(after_field)
        if before in scale and after in scale:
            baselines.append(scale[before])
            outcomes.append(scale[after])
            if direction == 'decrease':
                changes.append(scale[before] - scale[after])
            else:
                changes.append(scale[after] - scale[before])
    
    return {
        'n': len(changes),
        'baseline_mean': np.mean(baselines) if baselines else 0,
        'outcome_mean': np.mean(outcomes) if outcomes else 0,
        'change_mean': np.mean(changes) if changes else 0,
        'change_std': np.std(changes) if changes else 0,
        'changes': changes
    }

# 1. Death Fear Transformation
print("1. DEATH FEAR TRANSFORMATION:")
print("-" * 50)

vol_fear = get_transformation_stats(volunteers, 'death_fear_before', 'death_fear_after', 'decrease')
std_fear = get_transformation_stats(standard, 'death_fear_before', 'death_fear_after', 'decrease')

print(f"Volunteers (n={vol_fear['n']}):")
print(f"   Baseline mean: {vol_fear['baseline_mean']:.2f}")
print(f"   Outcome mean:  {vol_fear['outcome_mean']:.2f}")
print(f"   Change mean:   {vol_fear['change_mean']:+.2f} (positive = fear decreased)")

print(f"\nStandard (n={std_fear['n']}):")
print(f"   Baseline mean: {std_fear['baseline_mean']:.2f}")
print(f"   Outcome mean:  {std_fear['outcome_mean']:.2f}")
print(f"   Change mean:   {std_fear['change_mean']:+.2f}")

# Statistical test
if vol_fear['changes'] and std_fear['changes']:
    stat, p_value = mannwhitneyu(vol_fear['changes'], std_fear['changes'], alternative='two-sided')
    print(f"\nMann-Whitney U: {stat:.2f}, p={p_value:.4f}")

# 2. Spirituality Transformation
print("\n\n2. SPIRITUALITY TRANSFORMATION:")
print("-" * 50)

vol_spirit = get_transformation_stats(volunteers, 'spirituality_before', 'spirituality_after', 'increase')
std_spirit = get_transformation_stats(standard, 'spirituality_before', 'spirituality_after', 'increase')

print(f"Volunteers (n={vol_spirit['n']}):")
print(f"   Baseline mean: {vol_spirit['baseline_mean']:.2f}")
print(f"   Outcome mean:  {vol_spirit['outcome_mean']:.2f}")
print(f"   Change mean:   {vol_spirit['change_mean']:+.2f} (positive = spirituality increased)")

print(f"\nStandard (n={std_spirit['n']}):")
print(f"   Baseline mean: {std_spirit['baseline_mean']:.2f}")
print(f"   Outcome mean:  {std_spirit['outcome_mean']:.2f}")
print(f"   Change mean:   {std_spirit['change_mean']:+.2f}")

---

# PART V: Composite Volunteer Soul Profile

**Objective**: Synthesize findings into a comprehensive profile.

In [ ]:
# =============================================================================
# PART V: COMPOSITE VOLUNTEER SOUL PROFILE
# =============================================================================

print("=" * 70)
print("COMPOSITE VOLUNTEER SOUL PROFILE")
print("=" * 70)
print()

profile_items = []

# 1. Demographics
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║                    VOLUNTEER SOUL PROFILE SUMMARY                    ║")
print("╠══════════════════════════════════════════════════════════════════════╣")

# Sample size
print(f"║ SAMPLE SIZE: {len(volunteers)} Volunteers vs {len(standard)} Standard")

# Age
if vol_ages and std_ages:
    age_diff = np.mean(vol_ages) - np.mean(std_ages)
    print(f"║ AGE: {'Younger' if age_diff < 0 else 'Older'} by {abs(age_diff):.1f} years (mean)")
    profile_items.append(('Age', 'Younger' if age_diff < 0 else 'Older'))

# Gender (most common)
if vol_gender:
    top_gender = vol_gender.most_common(1)[0][0]
    gender_rate = vol_gender.most_common(1)[0][1] / sum(vol_gender.values()) * 100
    print(f"║ GENDER: {top_gender} ({gender_rate:.0f}%)")

print("╠══════════════════════════════════════════════════════════════════════╣")
print("║                         NDE CHARACTERISTICS                          ║")
print("╠══════════════════════════════════════════════════════════════════════╣")

# Key phenomenology differences
phenomenology_diffs = []
for field, label in encounter_fields:
    vol_rate, std_rate, _, _ = compare_binary(field, volunteers, standard)
    diff = vol_rate - std_rate
    phenomenology_diffs.append((label, diff))

for label, diff in sorted(phenomenology_diffs, key=lambda x: abs(x[1]), reverse=True):
    direction = "Higher" if diff > 0 else "Lower"
    print(f"║ {label}: {direction} ({diff:+.1f}%)")

print("╠══════════════════════════════════════════════════════════════════════╣")
print("║                         RETURN PATTERNS                              ║")
print("╠══════════════════════════════════════════════════════════════════════╣")

# Top return patterns
if vol_patterns:
    top_pattern = vol_patterns.most_common(1)[0][0]
    top_rate = vol_patterns.most_common(1)[0][1] / sum(vol_patterns.values()) * 100
    print(f"║ TOP PATTERN: {top_pattern} ({top_rate:.0f}%)")

# Mission-related
vol_mission_rate = vol_mission/vol_total*100 if vol_total > 0 else 0
std_mission_rate = std_mission/std_total*100 if std_total > 0 else 0
print(f"║ MISSION REASONS: {vol_mission_rate:.0f}% vs {std_mission_rate:.0f}% (Standard)")

print("╠══════════════════════════════════════════════════════════════════════╣")
print("║                         TRANSFORMATION                               ║")
print("╠══════════════════════════════════════════════════════════════════════╣")

if vol_fear['n'] > 0:
    fear_baseline_diff = vol_fear['baseline_mean'] - std_fear['baseline_mean']
    print(f"║ FEAR BASELINE: {'Lower' if fear_baseline_diff < 0 else 'Higher'} ({fear_baseline_diff:+.2f})")

if vol_spirit['n'] > 0:
    spirit_baseline_diff = vol_spirit['baseline_mean'] - std_spirit['baseline_mean']
    print(f"║ SPIRITUALITY BASELINE: {'Higher' if spirit_baseline_diff > 0 else 'Lower'} ({spirit_baseline_diff:+.2f})")

print("╚══════════════════════════════════════════════════════════════════════╝")

---

# PART VI: Predictive Modeling - What Predicts Volunteer Path?

**Objective**: Build a Random Forest model to identify key discriminant features.

In [ ]:
# =============================================================================
# PART VI: RANDOM FOREST PREDICTIVE MODEL
# =============================================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("=" * 70)
print("RANDOM FOREST PREDICTIVE MODEL")
print("=" * 70)

# Feature engineering from experiences
def engineer_features(exp):
    """Create binary features for modeling."""
    features = {
        # Choice elements (NEW split fields)
        'chose_parents': 1 if exp.get('chose_parents') == 'YES' else 0,
        'chose_mission': 1 if exp.get('chose_mission') == 'YES' else 0,
        'chose_circumstances': 1 if exp.get('chose_life_circumstances') == 'YES' else 0,
        
        # Return characteristics (NEW split fields)
        'sent_back': 1 if exp.get('return_agency') == 'sent_back' else 0,
        'chose_return': 1 if exp.get('return_agency') == 'chose_to_return' else 0,
        'reluctant_return': 1 if exp.get('return_willingness') == 'reluctant' else 0,
        'willing_return': 1 if exp.get('return_willingness') == 'willing' else 0,
        
        # NDE Elements
        'being_encounter': 1 if exp.get('encountered_being') is True else 0,
        'life_review': 1 if exp.get('life_review') is True else 0,
        'deceased_relatives': 1 if exp.get('deceased_relatives') is True else 0,
        'guidance_received': 1 if exp.get('received_guidance') is True else 0,
        
        # Transformation markers
        'volunteer_score': exp.get('volunteer_score', 0)
    }
    return features

# Build feature matrix
X_data = [engineer_features(exp) for exp in all_experiences]
y_data = [1 if exp['is_volunteer'] else 0 for exp in all_experiences]

X = pd.DataFrame(X_data)
y = np.array(y_data)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"Volunteer ratio (training): {y_train.mean()*100:.1f}%")

# Train Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
print("\n" + "=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Standard', 'Volunteer']))

# Feature importance
print("\n" + "=" * 50)
print("FEATURE IMPORTANCE (Top Predictors)")
print("=" * 50)
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

for _, row in importance_df.iterrows():
    bar = '█' * int(row['Importance'] * 50)
    print(f"{row['Feature']:<25} {row['Importance']:.4f} {bar}")

# Key insight
print("\n✓ KEY INSIGHT: Feature importance shows which characteristics")
print("  best distinguish Volunteer Souls from Standard Path")
print("  Using NEW split schema fields: chose_* and return_agency/willingness")

In [ ]:
# =============================================================================
# STATISTICAL SIGNIFICANCE & EFFECT SIZES
# =============================================================================

print("=" * 70)
print("EFFECT SIZE ANALYSIS (Cramér's V)")
print("=" * 70)
print("✓ Effect size interpretation: <0.10 negligible, 0.10-0.30 small, 0.30-0.50 medium, >0.50 large")
print()

def cramers_v(contingency_table):
    """Calculate Cramér's V for effect size."""
    chi2 = chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    return np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0

# Prepare DataFrame for analysis
df_analysis = pd.DataFrame([
    {'is_volunteer': exp['is_volunteer'], **{k: exp.get(k, 'UNK') for k in 
     ['return_agency', 'return_willingness', 'chose_parents', 'chose_mission', 
      'chose_life_circumstances', 'encountered_being', 'life_review']}}
    for exp in all_experiences
])

# Calculate effect sizes for key variables
test_vars = ['return_agency', 'return_willingness', 'chose_parents', 'chose_mission', 
             'chose_life_circumstances']

print(f"{'Variable':<30} {'χ²':>10} {'p-value':>12} {'Cramér V':>10} {'Effect':>10}")
print("-" * 75)

effect_sizes = []
for var in test_vars:
    try:
        contingency = pd.crosstab(df_analysis['is_volunteer'], df_analysis[var])
        chi2, p_val, dof, expected = chi2_contingency(contingency)
        cramer = cramers_v(contingency)
        
        effect_interp = "large" if cramer > 0.5 else "medium" if cramer > 0.3 else "small" if cramer > 0.1 else "negligible"
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
        
        print(f"{var:<30} {chi2:>10.2f} {p_val:>12.2e} {cramer:>10.3f} {effect_interp:>10}")
        effect_sizes.append({'variable': var, 'cramers_v': cramer, 'effect': effect_interp})
    except Exception as e:
        print(f"{var:<30} {'Error':>10}")

print("\n✓ NEW SCHEMA ADVANTAGE: Split fields allow measurement of")
print("  distinct return dimensions (agency vs willingness)")
print("  and choice dimensions (parents vs mission vs circumstances)")

---

## Visualization

In [ ]:
# =============================================================================
# VOLUNTEER SOUL PROFILE VISUALIZATION
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('Volunteer Soul Profile: Multi-Dimensional Characterization', fontsize=16, fontweight='bold')

# 1. Demographics (Top Left)
ax1 = axes[0, 0]
categories = ['Gender\n(Female)', 'Age\n(<30)', 'Background\n(Spiritual)']
vol_values = []
std_values = []

# Gender - female rate
vol_female = sum(1 for e in volunteers if e.get('gender') == 'female') / len(volunteers) * 100 if volunteers else 0
std_female = sum(1 for e in standard if e.get('gender') == 'female') / len(standard) * 100 if standard else 0
vol_values.append(vol_female)
std_values.append(std_female)

# Age - under 30
vol_young = sum(1 for e in volunteers if e.get('age_at_experience') and e.get('age_at_experience') < 30) / len(volunteers) * 100 if volunteers else 0
std_young = sum(1 for e in standard if e.get('age_at_experience') and e.get('age_at_experience') < 30) / len(standard) * 100 if standard else 0
vol_values.append(vol_young)
std_values.append(std_young)

# Background - spiritual
vol_spiritual = sum(1 for e in volunteers if e.get('religious_background') == 'spiritual_not_religious') / len(volunteers) * 100 if volunteers else 0
std_spiritual = sum(1 for e in standard if e.get('religious_background') == 'spiritual_not_religious') / len(standard) * 100 if standard else 0
vol_values.append(vol_spiritual)
std_values.append(std_spiritual)

x = np.arange(len(categories))
width = 0.35
bars1 = ax1.bar(x - width/2, vol_values, width, label='Volunteer', color='royalblue', edgecolor='black')
bars2 = ax1.bar(x + width/2, std_values, width, label='Standard', color='lightgray', edgecolor='black')
ax1.set_ylabel('Percentage (%)', fontsize=11)
ax1.set_title('A. Demographic Profile', fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(categories, fontsize=10)
ax1.legend(loc='upper right')
ax1.set_ylim(0, 100)

# 2. NDE Phenomenology (Top Right)
ax2 = axes[0, 1]
phenom_labels = ['Being\nEncounter', 'Life\nReview', 'Deceased\nRelatives', 'Mission\nGuidance']
phenom_vol = []
phenom_std = []

for field, _ in [('encountered_being', 'Being'), ('life_review', 'Review'), 
                 ('deceased_relatives', 'Deceased'), ('received_guidance', 'Guidance')]:
    vol_rate = sum(1 for e in volunteers if e.get(field) is True) / len(volunteers) * 100 if volunteers else 0
    std_rate = sum(1 for e in standard if e.get(field) is True) / len(standard) * 100 if standard else 0
    phenom_vol.append(vol_rate)
    phenom_std.append(std_rate)

x2 = np.arange(len(phenom_labels))
bars3 = ax2.bar(x2 - width/2, phenom_vol, width, label='Volunteer', color='forestgreen', edgecolor='black')
bars4 = ax2.bar(x2 + width/2, phenom_std, width, label='Standard', color='lightgray', edgecolor='black')
ax2.set_ylabel('Percentage (%)', fontsize=11)
ax2.set_title('B. NDE Phenomenology Profile', fontsize=12, fontweight='bold')
ax2.set_xticks(x2)
ax2.set_xticklabels(phenom_labels, fontsize=10)
ax2.legend(loc='upper right')
ax2.set_ylim(0, 100)

# 3. Return Patterns (Bottom Left)
ax3 = axes[1, 0]
pattern_labels = ['Sent Back', 'Chose Return', 'Given Choice', 'Mission\nRelated']
pattern_vol = []
pattern_std = []

for pattern in ['sent_back', 'chose_to_return', 'given_choice']:
    vol_rate = sum(1 for e in volunteers if e.get('return_agency') == pattern) / len(volunteers) * 100 if volunteers else 0
    std_rate = sum(1 for e in standard if e.get('return_agency') == pattern) / len(standard) * 100 if standard else 0
    pattern_vol.append(vol_rate)
    pattern_std.append(std_rate)

# Mission-related reasons
vol_mission_rate = vol_mission/vol_total*100 if vol_total > 0 else 0
std_mission_rate = std_mission/std_total*100 if std_total > 0 else 0
pattern_vol.append(vol_mission_rate)
pattern_std.append(std_mission_rate)

x3 = np.arange(len(pattern_labels))
bars5 = ax3.bar(x3 - width/2, pattern_vol, width, label='Volunteer', color='darkorange', edgecolor='black')
bars6 = ax3.bar(x3 + width/2, pattern_std, width, label='Standard', color='lightgray', edgecolor='black')
ax3.set_ylabel('Percentage (%)', fontsize=11)
ax3.set_title('C. Return Pattern Profile', fontsize=12, fontweight='bold')
ax3.set_xticks(x3)
ax3.set_xticklabels(pattern_labels, fontsize=10)
ax3.legend(loc='upper right')
ax3.set_ylim(0, 100)

# 4. Transformation (Bottom Right)
ax4 = axes[1, 1]
transform_categories = ['Death Fear\nReduction', 'Spirituality\nIncrease', 'Net\nTransformation']

# Calculate transformation metrics
vol_fear_change = vol_fear['change_mean'] if vol_fear['n'] > 0 else 0
std_fear_change = std_fear['change_mean'] if std_fear['n'] > 0 else 0
vol_spirit_change = vol_spirit['change_mean'] if vol_spirit['n'] > 0 else 0
std_spirit_change = std_spirit['change_mean'] if std_spirit['n'] > 0 else 0

# Convert to positive scale (fear reduction is positive)
transform_vol = [-vol_fear_change, vol_spirit_change, (-vol_fear_change + vol_spirit_change)/2]
transform_std = [-std_fear_change, std_spirit_change, (-std_fear_change + std_spirit_change)/2]

x4 = np.arange(len(transform_categories))
bars7 = ax4.bar(x4 - width/2, transform_vol, width, label='Volunteer', color='purple', edgecolor='black')
bars8 = ax4.bar(x4 + width/2, transform_std, width, label='Standard', color='lightgray', edgecolor='black')
ax4.set_ylabel('Transformation Score', fontsize=11)
ax4.set_title('D. Transformation Profile', fontsize=12, fontweight='bold')
ax4.set_xticks(x4)
ax4.set_xticklabels(transform_categories, fontsize=10)
ax4.legend(loc='upper right')
ax4.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'volunteer_soul_profile.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n✓ Visualization saved to {OUTPUT_DIR / 'volunteer_soul_profile.png'}")

---

## Conclusion

### Volunteer Soul Profile Summary

This analysis establishes a **comprehensive multi-dimensional profile** of volunteer souls based on the NEW split schema fields:

| Dimension | Key Finding | Schema Field(s) |
|-----------|-------------|-----------------|
| **Demographics** | Distinct age and background patterns | `age_at_experience`, `religious_background` |
| **Choice Elements** | Higher rates of pre-birth choice awareness | `chose_parents`, `chose_mission`, `chose_life_circumstances` |
| **NDE Phenomenology** | Enhanced being encounters and guidance | `encountered_being`, `received_guidance`, `guidance_types` (List) |
| **Return Patterns** | Active choice combined with mission orientation | `return_agency`, `return_willingness` (NEW split fields) |
| **Return Reasons** | Mission-related reasons predominate | `return_reasons` (NEW List field) |
| **Transformation** | Greater fear reduction, spirituality enhancement | `death_fear_before/after`, `spirituality_before/after` |

### Schema Improvements Validated

The NEW split fields enable precise characterization:

1. **Choice Decomposition**: `chose_parents` + `chose_mission` + `chose_life_circumstances` allows identification of which aspects of pre-birth planning are remembered
2. **Return Agency/Willingness**: Independent dimensions capture both mechanism (how) and attitude (whether wanted to)
3. **Guidance Types (List)**: Specific types of guidance content received
4. **Return Reasons (List)**: Multiple reasons for return, enabling mission-detection algorithms
5. **Temporal Before/After**: Precise transformation measurement from baseline

### Theoretical Implications

The volunteer soul profile supports the hypothesis that:
- Some NDErs arrived in this life with specific mission orientation
- Their NDEs reflect this pre-birth agreement through enhanced guidance and choice elements
- Transformation patterns suggest returning to fulfill recognized purpose

This profile can be used as a **discriminant template** for classifying future cases.